# OCR Diagram Pipeline Notebook

This notebook version of the project runs the same staged pipeline as `main.py`, while making each step easier to configure, inspect, and rerun.

The pipeline can run in three modes:

- `pipeline`: OCR, grouping, OpenCV connection detection, optional local LLM normalization, then output files.
- `direct-llm`: send rendered diagram pages directly to a vision-capable local LLM endpoint.
- `both`: run both paths for comparison.

Raw diagram images are only sent to the LLM in `direct-llm` mode. The staged `pipeline` mode sends structured OCR and geometry evidence only.

## 1. Environment Check

Run this cell first. The project expects Python 3.11, 3.12, or 3.13 because the PaddlePaddle dependency stack does not currently support Python 3.14+.

In [ ]:
from __future__ import annotations

import importlib.util
import json
import shutil
import sys
import traceback
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "diagram_parser").exists():
    raise RuntimeError(f"Run this notebook from the project root, not {PROJECT_ROOT}")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

version = sys.version_info[:2]
if not ((3, 11) <= version < (3, 14)):
    raise RuntimeError(
        f"Unsupported Python runtime: {sys.version.split()[0]}. "
        "Use Python 3.11, 3.12, or 3.13."
    )

required_modules = ["paddle", "paddleocr", "pypdfium2", "cv2", "numpy", "requests"]
missing = [name for name in required_modules if importlib.util.find_spec(name) is None]

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.version.split()[0]}")
print(f"Missing modules: {missing or 'none'}")
print(f"Mermaid CLI available: {bool(shutil.which('mmdc'))}")

If dependencies are missing, install them in the environment backing this notebook kernel:

```bash
pip install -r requirements.txt
```

Optional Mermaid SVG rendering requires `mmdc` from Mermaid CLI. The pipeline still writes `topology.mmd` when `mmdc` is unavailable.

## 2. Configure Inputs

Set `INPUT_PATH` to a PNG, JPG, JPEG, or PDF infrastructure diagram. Start with `SKIP_LLM = True` when you want deterministic OCR/grouping output without running a local model.

In [ ]:
# Required input
INPUT_PATH = Path("/path/to/diagram.png")  # replace with your PNG/JPG/PDF path

# Output location. Leave as None to use PROJECT_ROOT / "output".
OUTPUT_DIR = None

# Execution mode: "pipeline", "direct-llm", or "both".
MODE = "both"

# Deterministic first pass. Set False to use the configured local LLM endpoint.
SKIP_LLM = False
ALLOW_LLM_FALLBACK = True

# Local LLM settings used when SKIP_LLM is False, or when MODE includes direct-llm.
MODEL = "qwen/qwen3.5-35b-a3b"
BASE_URL = "http://127.0.0.1:1234/v1"
TIMEOUT_SECONDS = 300

# Optional uControl metadata. Leave as None to infer the application name.
APPLICATION_NAME = "CODA"
APP_ID = None
SAVE_UCONTROL_ASSET_TAGS = True
INCLUDE_UCONTROL_RAG = True

# OCR/PDF tuning.
PDF_SCALE = 1.25
MAX_PAGES = None
OCR_IMAGE_PADDING = 40
OCR_IMAGE_SCALE = 1.0
USE_OCR_CACHE = True
REFRESH_OCR_CACHE = False

# Optional local PaddleOCR model directories for offline runs.
OCR_DET_MODEL_DIR = None
OCR_REC_MODEL_DIR = None
OCR_CLS_MODEL_DIR = None

# Grouping and connection tuning.
GROUP_DISTANCE = 30.0
ENDPOINT_DISTANCE = 80.0

input_path = INPUT_PATH.expanduser().resolve()
output_root = Path(OUTPUT_DIR).expanduser().resolve() if OUTPUT_DIR else PROJECT_ROOT / "output"

print(f"Input: {input_path}")
print(f"Output root: {output_root}")
if not input_path.exists():
    print("Update INPUT_PATH before running the pipeline.")

## 3. Build Pipeline Configuration

In [ ]:
from diagram_parser.config import PipelineConfig

config = PipelineConfig()

config.llm.enabled = not SKIP_LLM
config.llm.allow_fallback_on_error = ALLOW_LLM_FALLBACK
config.llm.include_ucontrol_asset_rag = INCLUDE_UCONTROL_RAG
config.llm.model = MODEL
config.llm.base_url = BASE_URL
config.llm.timeout_seconds = TIMEOUT_SECONDS

config.output.application_name = APPLICATION_NAME
config.output.app_id = APP_ID
config.output.save_ucontrol_asset_tags = SAVE_UCONTROL_ASSET_TAGS

config.ocr.pdf_render_scale = PDF_SCALE
config.ocr.max_pages = MAX_PAGES
config.ocr.image_padding_pixels = OCR_IMAGE_PADDING
config.ocr.image_scale = OCR_IMAGE_SCALE
config.ocr.use_cache = USE_OCR_CACHE
config.ocr.refresh_cache = REFRESH_OCR_CACHE
config.ocr.text_detection_model_dir = OCR_DET_MODEL_DIR
config.ocr.text_recognition_model_dir = OCR_REC_MODEL_DIR
config.ocr.textline_orientation_model_dir = OCR_CLS_MODEL_DIR

config.grouping.max_merge_distance = GROUP_DISTANCE
config.connections.node_endpoint_distance = ENDPOINT_DISTANCE

config

## 4. Run the Selected Mode

This cell mirrors the CLI mode handling in `main.py`. Outputs are written under `output/pipeline` and/or `output/direct_llm`.

In [ ]:
from diagram_parser.main import run_direct_llm, run_pipeline

if MODE not in {"pipeline", "direct-llm", "both"}:
    raise ValueError("MODE must be 'pipeline', 'direct-llm', or 'both'")
if MODE != "pipeline" and SKIP_LLM:
    raise ValueError("SKIP_LLM can only be used with MODE='pipeline'")
if not input_path.exists():
    raise FileNotFoundError(f"Input file does not exist: {input_path}")
if not input_path.is_file():
    raise FileNotFoundError(f"Input path is not a file: {input_path}")

results = []
failures = []

def run_mode(label, runner, mode_output_dir):
    try:
        topology, output_paths = runner(
            image_path=input_path,
            output_dir=mode_output_dir,
            config=config,
        )
        results.append((label, topology, output_paths))
    except Exception as exc:
        failures.append((label, exc, traceback.format_exc()))

if MODE in {"pipeline", "both"}:
    run_mode("pipeline", run_pipeline, output_root / "pipeline")

if MODE in {"direct-llm", "both"}:
    run_mode("direct_llm", run_direct_llm, output_root / "direct_llm")

if not results and failures:
    label, exc, details = failures[0]
    print(details)
    raise RuntimeError(f"All selected modes failed. First failure: [{label}] {exc}")

for label, topology, output_paths in results:
    print(f"[{label}] Detected {len(topology.nodes)} nodes and {len(topology.edges)} edges")
    for name, path in output_paths.items():
        print(f"[{label}] {name}: {path}")

for label, exc, details in failures:
    print(f"[{label}] failed: {exc}")

## 5. Inspect Topology JSON

In [ ]:
def load_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))

for label, topology, output_paths in results:
    topology_json = load_json(output_paths["json"])
    print(f"\n=== {label}: topology.json ===")
    print(json.dumps(topology_json, indent=2))

## 6. Inspect Intermediate Candidates

The staged pipeline writes OCR spans, grouped candidate nodes, and candidate connections. These files are useful when tuning grouping and connection thresholds.

In [ ]:
for label, topology, output_paths in results:
    intermediate_path = output_paths.get("intermediate")
    if not intermediate_path or not intermediate_path.exists():
        continue

    structured = load_json(intermediate_path)
    print(f"\n=== {label}: structured candidates ===")
    print(f"Pages: {len(structured.get('pages', []))}")
    print(f"OCR spans: {len(structured.get('ocr_spans', []))}")
    print(f"Candidate nodes: {len(structured.get('candidate_nodes', []))}")
    print(f"Candidate connections: {len(structured.get('candidate_connections', []))}")
    print("\nCandidate nodes:")
    for node in structured.get("candidate_nodes", []):
        print(f"- {node['node_id']}: {node['label']} [{node['type_hint']}] {node.get('type_reason') or ''}")
    print("\nCandidate connections:")
    for edge in structured.get("candidate_connections", []):
        hints = ", ".join(edge.get("label_hints", []))
        print(f"- {edge['from_node_id']} -> {edge['to_node_id']} confidence={edge['confidence']} hints={hints}")

## 7. Display Mermaid Output

If Mermaid CLI is installed, the project may also write a rendered SVG. Otherwise, use the `.mmd` source with a Mermaid-capable viewer.

In [ ]:
from IPython.display import SVG, Markdown, display

for label, topology, output_paths in results:
    print(f"\n=== {label}: Mermaid ===")
    mermaid_source = output_paths["mermaid"].read_text(encoding="utf-8")
    display(Markdown(f"```mermaid\n{mermaid_source}\n```"))

    svg_path = output_paths.get("mermaid_svg")
    if svg_path and svg_path.exists():
        display(SVG(filename=str(svg_path)))

## 8. Inspect uControl Output

These files are written when `SAVE_UCONTROL_ASSET_TAGS = True`.

In [ ]:
ucontrol_keys = [
    "ucontrol_model_create",
    "ucontrol_populate_umap",
    "ucontrol_retrieval_requests",
]

for label, topology, output_paths in results:
    print(f"\n=== {label}: uControl files ===")
    for key in ucontrol_keys:
        path = output_paths.get(key)
        if not path or not path.exists():
            print(f"{key}: not written")
            continue
        print(f"\n{key}: {path}")
        print(json.dumps(load_json(path), indent=2)[:4000])

## 9. Troubleshooting

- If PaddleOCR cannot download models, set the `OCR_*_MODEL_DIR` variables to local model directories.
- If a local LLM server rejects strict structured output, the project client retries without strict response-format settings.
- For large PDFs, lower `PDF_SCALE` and set `MAX_PAGES = 1` while tuning.
- To rerun OCR instead of reusing `ocr_spans.json`, set `REFRESH_OCR_CACHE = True`.
- For deterministic output only, keep `SKIP_LLM = True`.